#### [Notebook 2 Quick Start](https://confluence.shopee.io/x/7O0lhw)
#### Common Use Cases and Solution Using Notebook 2
- [Multi Kernels in Notebook 2](https://confluence.shopee.io/x/4Od2hw)
- [Execute HDFS command](https://confluence.shopee.io/x/P5w8hw)
- [Browse, upload and download from Data Infra S3](https://confluence.shopee.io/x/xo9Ohw)
- [Setting up 3rd party packages for your Notebook 2](https://confluence.shopee.io/x/JZFOhw)
- [Develop and setup user defined function or library](https://confluence.shopee.io/x/hpjehg)
- [Execute Presto/Trino query directly in Python Kernel Notebook](https://confluence.shopee.io/x/WpMlhw)

> Note: "Python on K8S" Kernel cannot execute Spark functions. ***If you want to use Spark, change kernel to Spark***. [How to change Kernel](https://confluence.shopee.io/x/4Od2hw#MultiKernelsinNotebook2-SetKernelwhencreatingnewNotebook2file)

In [ ]:
#!/usr/bin/python3
# encoding: utf-8
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
import pandas as pd
import re,json,sys,os
import random
import math
import datetime
import argparse


spark = SparkSession.builder \
            .appName('init_us_risk_table') \
            .master('yarn') \
            .config('spark.sql.execution.arrow.pyspark.enabled', True) \
            .config('spark.sql.session.timeZone', 'UTC') \
            .config('spark.driver.memory','16G') \
            .config('spark.executor.memory', '16g') \
            .config('spark.executor.cores', '4') \
            .config('spark.num.executors', '64') \
            .config('spark.ui.showConsoleProgress', True) \
            .config('hive.exec.dynamic.partition.mode', 'nonstrict') \
            .config('spark.sql.repl.eagerEval.enabled', True) \
            .enableHiveSupport() \
            .getOrCreate()

storage_options = {
    # The credentials are not required to be set if they are already set in the environment variables / credentials file.
    "key": "risk_feature",
    "secret": "bSc1Ua7jqqIw",
    # for SG idc, use this endpoint
    # for others can refer https://confluence.shopee.io/display/DI/3.+DI+S3+Usage
    "client_kwargs": {"endpoint_url": "https://s3g.data-infra.shopee.io"}
}

grass_date = "${grass_date}"
sql_2bscore = """
  select 
 task_id 
 ,row_number() over ( partition by task_id order by backtest_date ) as  row_num
 ,backtest_date 
 ,ktp_no ,phone_no 
 ,hit_list_type 
 ,reason_code 
 ,case when is_ktp_match = 0 and is_phone_match = 0 then 'no match' when score is null then 'no score' else score end as score
 ,case when is_ktp_match = 0 and is_phone_match = 0 then 'no match' when score_v3 is null then 'no score' else score_v3 end as score_v3 
 from risk_feature.ads_2bscore_id_ktp_phone_backtesting_info_rt
 where grass_date = '{grass_date}'
 order by id asc
 """.format(grass_date=grass_date)

sql_multihead = """select task_id,backtest_date,ktp_no,phone_no,mh_latest_query_to_now_days,mh_first_query_to_now_days,mh_query_cnt_7d,mh_query_cnt_30d,mh_query_cnt_90d,mh_query_cnt_180d,mh_query_cnt_360d,mh_query_cnt_720d,mh_institution_cnt_7d,mh_institution_cnt_30d,mh_institution_cnt_90d,mh_institution_cnt_180d,mh_institution_cnt_360d,mh_institution_cnt_720d,mh_attempted_institutions_day_max_7d,mh_attempted_institutions_day_max_30d,mh_attempted_institutions_day_max_90d,mh_attempted_institutions_day_max_180d,mh_attempted_institutions_day_max_360d,mh_attempted_institutions_day_max_720d,mh_attempted_institutions_month_max_90d,mh_attempted_institutions_month_max_180d,mh_attempted_institutions_month_max_360d,mh_attempted_institutions_month_max_720d,mh_consecutive_query_day_min_7d,mh_consecutive_query_day_min_30d,mh_consecutive_query_day_min_90d,mh_consecutive_query_day_min_180d,mh_consecutive_query_day_min_360d,mh_consecutive_query_day_min_720d 
from risk_feature.ads_risk_id_multihead_ktp_phone_backtesting_info_rt where grass_date = '{grass_date}'
order by id asc
""".format(grass_date=grass_date)

print(sql_2bscore)
print(sql_multihead)
df_2bscore = spark.sql(sql_2bscore)
df_multihead = spark.sql(sql_multihead)

pandas_df_2bscore = df_2bscore.toPandas()
pandas_df_multihead = df_multihead.toPandas()


column_mapping = {
    "row_num":"No",
    "backtest_date":"Backtest date",
    "ktp_no":"KTP",
    "phone_no":"Nomor Telepon",
    "hit_list_type":"Blacklist (1) or Greylist (2)",
    "reason_code":"Reason Code",
    "score":"ST Score",
    "score_v3":"ST Score V3"
    # 添加其他的列名映射
}

task_id_list_2bscore = pandas_df_2bscore['task_id'].unique()
task_id_list_multihead = pandas_df_multihead['task_id'].unique()

for task_id in task_id_list_2bscore:
    task_date = task_id.split("_")[0]
    s3_path = "s3://sg-risk-feature-data/ID/ads_2bscore_id_ktp_phone_backtesting_info_df/{task_date}/{task_id}/{task_id}_001.csv".format(task_id=task_id, task_date=task_date)
    print("s3_path-->",s3_path)
    task_df_2bscore = pandas_df_2bscore[pandas_df_2bscore['task_id'] == task_id]
    task_df_2bscore_result = task_df_2bscore[['row_num', 'backtest_date','ktp_no','phone_no','hit_list_type','reason_code','score','score_v3']].rename(columns=column_mapping)
    task_df_2bscore_result.to_csv(s3_path, storage_options=storage_options, index=False)
    
    # 为当前task_id生成_SUCCESS文件
    success_file_path = "s3://sg-risk-feature-data/ID/ads_2bscore_id_ktp_phone_backtesting_info_df/{task_date}/{task_id}/_SUCCESS".format(task_id=task_id, task_date=task_date)
    pd.DataFrame().to_csv(success_file_path, storage_options=storage_options, index=False, header=False)
    print(f"_SUCCESS file created for task_id: {task_id}")

for task_id in task_id_list_multihead:
    task_date = task_id.split("_")[0]
    s3_path = "s3://sg-risk-feature-data/ID/ads_2bscore_id_ktp_phone_backtesting_info_df/{task_date}/{task_id}/{task_id}_001.csv".format(task_id=task_id, task_date=task_date)
    print("s3_path-->",s3_path)
    task_df_multihead = pandas_df_multihead[pandas_df_multihead['task_id'] == task_id]
    task_df_multihead.to_csv(s3_path, storage_options=storage_options, index=False)
    
    # 为当前task_id生成_SUCCESS文件
    success_file_path = "s3://sg-risk-feature-data/ID/ads_2bscore_id_ktp_phone_backtesting_info_df/{task_date}/{task_id}/_SUCCESS".format(task_id=task_id, task_date=task_date)
    pd.DataFrame().to_csv(success_file_path, storage_options=storage_options, index=False, header=False)
    print(f"_SUCCESS file created for task_id: {task_id}")

